# freeze-requires-grad — ex1: freeze a toy backbone and collect trainable params

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `freeze-requires-grad`. Running the final beacon cell reports progress against the `PyTorch: freeze via requires_grad=False` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: freeze via requires_grad=False` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`freeze-requires-grad`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "freeze-requires-grad"
DD_SUBTOPIC = "PyTorch: freeze via requires_grad=False"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Freeze backbone via `requires_grad = False` — quick refresher

Transfer learning: take a pretrained model, *freeze* the backbone so gradients don't update it, and train only a new task-specific head. The freeze idiom:

```
for p in model.parameters():
    p.requires_grad = False        # freeze everything
model.fc = nn.Linear(in_features, n_classes)   # new head — defaults to requires_grad=True
```

Or all-at-once via the helper method:

```
model.requires_grad_(False)        # freeze recursively in-place
```

**What `requires_grad = False` does.** It tells autograd to skip computing gradients for that tensor during `backward()`. Forward passes still flow through the param normally — only the gradient computation is suppressed.

**Critical optimizer step.** Pass ONLY the trainable params to the optimizer:

```
trainable = [p for p in model.parameters() if p.requires_grad]
opt = t.optim.Adam(trainable, lr=...)
```

If you forget this and pass `model.parameters()`, Adam will still try to update frozen params using their (zero) gradients — wastes memory on moment buffers and produces a subtle warning.

**Why new modules unfreeze automatically.** Brand-new `nn.Linear` / `nn.Conv2d` instances have `requires_grad = True` by default — only the params you explicitly froze stay frozen. So replacing `model.fc` after the freeze is the standard pattern: head trainable, backbone frozen, no extra wiring.

### Exercise 1 — freeze a toy backbone and collect trainable params

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the transfer-learning freeze pattern: set `requires_grad = False` on every backbone param, replace the head, and collect the (still-trainable) head params for the optimizer.
> Keywords: transfer-learning, requires_grad, freeze, head-replace
> ```

**KCs targeted:** `freeze-requires-grad-false`, `collect-trainable-params-for-optimizer`

A toy 'pretrained' model is provided:

```
class ToyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
        )
        self.fc = nn.Linear(16, 1000)   # ImageNet-style head
    def forward(self, x):
        return self.fc(self.encoder(x))
```

Implement `ex1_freeze_and_swap_head(model, n_classes)` that does the full transfer-learning prep, in this order:

1. **Freeze everything**: set `p.requires_grad = False` for every parameter currently in `model`.
2. **Replace the head**: set `model.fc = nn.Linear(16, n_classes)`. Brand-new linear modules default to `requires_grad=True`, so this re-unfreezes the head implicitly.
3. **Collect trainable params**: build `trainable_params = [p for p in model.parameters() if p.requires_grad]`.
4. Return a tuple `(model, trainable_params)`.

**What the test checks.**
- All encoder params have `requires_grad == False` after the call.
- The new head has `requires_grad == True` on both weight and bias.
- `trainable_params` contains exactly the new head's (weight + bias) — 2 tensors, totalling `n_classes * 16 + n_classes` scalars.
- An Adam optimizer built on `trainable_params` runs without warnings.
- A backward pass updates the head params but leaves encoder params bit-identical.

In [ ]:
import torch.nn as nn

class ToyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
        )
        self.fc = nn.Linear(16, 1000)
    def forward(self, x):
        return self.fc(self.encoder(x))


def ex1_freeze_and_swap_head(model, n_classes: int):
    """Freeze backbone, swap head to (16, n_classes), return (model, trainable_params)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn
    from torch.nn import functional as F

    model = ToyBackbone()
    n_classes = 5

    # Snapshot encoder params before the call — they must be unchanged after backward.
    enc_before = [p.detach().clone() for p in model.encoder.parameters()]

    model, trainable = ex1_freeze_and_swap_head(model, n_classes)

    # All encoder params must be frozen.
    for p in model.encoder.parameters():
        assert p.requires_grad is False, f'encoder param still trainable: {p.shape}'

    # The new head must have requires_grad=True on both weight + bias.
    assert model.fc.weight.requires_grad is True
    assert model.fc.bias.requires_grad is True
    # Head shape must be (n_classes, 16).
    assert model.fc.weight.shape == (n_classes, 16), (
        f'fc weight wrong shape: {tuple(model.fc.weight.shape)} expected ({n_classes}, 16)'
    )
    assert model.fc.bias.shape == (n_classes,)

    # trainable params: exactly 2 tensors (fc.weight + fc.bias), totaling 16*n + n scalars.
    assert len(trainable) == 2, f'expected 2 trainable tensors (fc.weight, fc.bias), got {len(trainable)}'
    total_trainable = sum(p.numel() for p in trainable)
    assert total_trainable == 16 * n_classes + n_classes, (
        f'trainable scalar count wrong: {total_trainable} vs {16*n_classes + n_classes}'
    )
    # Both trainable tensors must be the head params.
    trainable_ids = {id(p) for p in trainable}
    assert id(model.fc.weight) in trainable_ids
    assert id(model.fc.bias)  in trainable_ids

    # Optimizer construction must succeed.
    opt = t.optim.Adam(trainable, lr=1e-2)

    # Backward step: encoder params must remain bit-identical; head params must change.
    head_w_before = model.fc.weight.detach().clone()
    x = t.randn(4, 10)
    y = model(x)
    loss = F.cross_entropy(y, t.tensor([0, 1, 2, 3]))
    loss.backward()
    opt.step()

    # Encoder params unchanged (frozen).
    for p_now, p_then in zip(model.encoder.parameters(), enc_before):
        assert t.equal(p_now, p_then), 'frozen encoder param mutated by backward+step'
        assert p_now.grad is None or t.all(p_now.grad == 0), 'frozen param has nonzero grad'

    # Head changed.
    assert not t.equal(model.fc.weight, head_w_before), 'head weight did NOT update after step'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_freeze_and_swap_head(model, n_classes: int):
    import torch.nn as nn
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(16, n_classes)         # new module defaults to requires_grad=True
    trainable = [p for p in model.parameters() if p.requires_grad]
    return model, trainable
```

**Why the order matters.** If you replaced the head BEFORE freezing, the `for p in model.parameters()` loop would also freeze the brand-new head. Wrong order → no trainable params → Adam complains. Always freeze first, swap second.

**Equivalent freezes.**
- `model.requires_grad_(False)` — the in-place method, recursively visits children. Same effect.
- `for p in model.encoder.parameters(): p.requires_grad = False` — finer-grained, freeze only a submodule. ARENA's transfer-learning exercise uses this form to freeze only everything *except* `out_layers[-1]`.

**Why pass `trainable_params` (not `model.parameters()`) to Adam.** If you pass all params, Adam still allocates moment buffers for the frozen ones (wastes memory) and you'll see a `UserWarning` about gradients being None. Filtering up-front keeps the optimizer state lean.

**Why `.grad` is None (not zero) for frozen params.** Autograd never *creates* a `.grad` attribute on a tensor that doesn't have `requires_grad=True` — the gradient buffer simply never gets allocated. `.grad is None` is the canonical post-condition for a frozen param after backward; you do NOT need to `zero_grad()` them.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()